# T4D Response V2 - Phase 1 Foundation

Goal: load the hourly estuary history, attach clear station metadata, build one clean daily table, and save it as fresh material for the next notebook.

Outputs:
- `t4d.v2.phase1.daily_foundation.csv`
- `t4d.v2.phase1.station_metadata.csv`
- `t4d.v2.phase1.station_year_coverage.csv`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme( style = 'whitegrid' )
pd.set_option( 'display.max_columns', 120 )
pd.set_option( 'display.max_rows', 80 )

phase_out_dir = Path( '../data/reference/response_v2' )
phase_out_dir.mkdir( parents = True, exist_ok = True )

water_path = Path( '../data/1hr/t4d.1hr.water.history.csv' )
lookup_path = Path( '../data/reference/nerrs_station_index.csv' )

daily_out_path = phase_out_dir / 't4d.v2.phase1.daily_foundation.csv'
station_meta_out_path = phase_out_dir / 't4d.v2.phase1.station_metadata.csv'
station_year_out_path = phase_out_dir / 't4d.v2.phase1.station_year_coverage.csv'

In [ ]:
# Load the raw hourly history and the station lookup.
# Keep the rename map small and obvious so the daily table is easy to read later.
water = pd.read_csv( water_path )
station_lookup = pd.read_csv( lookup_path )

rename_map = { 
    'w_temp_c': 'water_temp',
    'w_sal_psu': 'salinity',
    'w_do_mg_l': 'oxygen',
    'w_do_pct': 'oxygen_sat',
    'depth_m': 'depth',
    'w_ph': 'ph',
    'm_wind_ms': 'wind_speed',
    'm_ssrd_kwh_m2': 'solar_radiation',
    'm_precip_mmh': 'precipitation',
    'm_temp_c': 'air_temp',
}
water = water.rename( columns = rename_map )

water[ 'region' ] = water[ 'region' ].astype( str ).str.strip( ).str.lower( )
water[ 'station' ] = water[ 'station' ].astype( str ).str.strip( ).str.lower( ).str[ -2: ]
water[ 'station_code' ] = water[ 'region' ] + water[ 'station' ]
water[ 'station_key' ] = water[ 'region' ] + '.' + water[ 'station' ]
water[ 'datetime' ] = pd.to_datetime( water[ 'datetime' ], errors = 'coerce' )
water = water.dropna( subset = [ 'datetime' ] ).copy( )

station_lookup[ 'region' ] = station_lookup[ 'region' ].astype( str ).str.strip( ).str.lower( )
station_lookup[ 'station' ] = station_lookup[ 'station' ].astype( str ).str.strip( ).str.lower( ).str[ -2: ]
station_lookup[ 'station_code' ] = station_lookup[ 'region' ] + station_lookup[ 'station' ]
station_lookup[ 'station_key' ] = station_lookup[ 'region' ] + '.' + station_lookup[ 'station' ]

lookup_keep_cols = [ 
    'station_key',
    'station_code',
    'region_name',
    'station_name',
    'latitude',
    'longitude',
    'in_t4d_1hr_water_history',
]
station_meta = station_lookup[ lookup_keep_cols ].drop_duplicates( subset = [ 'station_key' ] ).copy( )

water = water.merge( station_meta, on = [ 'station_key', 'station_code' ], how = 'left' )
water[ 'date' ] = water[ 'datetime' ].dt.floor( 'D' )

for col in [ 'station_key', 'station_code', 'region_name', 'station_name' ]:
    if col in water.columns:
        water[ col ] = water[ col ].astype( 'category' )

print( 'raw hourly rows:', len( water ) )
print( 'unique stations:', int( water[ 'station_key' ].nunique( ) ) )
display( water.head( 3 ) )

In [ ]:
# Collapse the hourly table to one row per station-day.
# Use plain daily means so the next notebooks can stay simple.
daily_metric_cols = [ 
    'water_temp',
    'salinity',
    'oxygen',
    'oxygen_sat',
    'ph',
    'depth',
    'air_temp',
    'wind_speed',
    'solar_radiation',
    'precipitation',
]
daily_metric_cols = [ col for col in daily_metric_cols if col in water.columns ]

daily = ( 
    water
    .groupby( 
        [ 
            'station_key',
            'station_code',
            'region',
            'station',
            'region_name',
            'station_name',
            'latitude',
            'longitude',
            'date',
        ],
        as_index = False,
    )[ daily_metric_cols ]
    .mean( )
)

daily = daily.sort_values( [ 'station_key', 'date' ] ).reset_index( drop = True )
daily[ 'year' ] = daily[ 'date' ].dt.year
daily[ 'month_num' ] = daily[ 'date' ].dt.month
daily[ 'is_warm_season' ] = daily[ 'month_num' ].between( 6, 9 )
daily[ 'doy_sin' ] = np.sin( 2 * np.pi * daily[ 'date' ].dt.dayofyear / 365.25 )
daily[ 'doy_cos' ] = np.cos( 2 * np.pi * daily[ 'date' ].dt.dayofyear / 365.25 )

baseline_start = pd.Timestamp( '1995-01-01' )
baseline_end = pd.Timestamp( '2014-12-31' )
baseline_source = daily.loc[ daily[ 'date' ].between( baseline_start, baseline_end ) ].copy( )

baseline_metric_cols = [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ]
baseline_metric_cols = [ col for col in baseline_metric_cols if col in daily.columns ]

baseline = ( 
    baseline_source
    .groupby( [ 'station_key' ], as_index = False )[ baseline_metric_cols ]
    .mean( )
    .rename( columns = { col: f'{ col }_baseline' for col in baseline_metric_cols } )
)

daily = daily.merge( baseline, on = 'station_key', how = 'left' )

for metric in baseline_metric_cols:
    daily[ f'delta_{ metric }' ] = daily[ metric ] - daily[ f'{ metric }_baseline' ]

station_year_coverage = ( 
    daily
    .groupby( [ 'station_key', 'station_code', 'region_name', 'station_name', 'year' ], as_index = False )
    .agg( 
        n_days = ( 'date', 'size' ),
        n_temp_days = ( 'water_temp', lambda values: int( values.notna( ).sum( ) ) ),
        n_oxygen_days = ( 'oxygen', lambda values: int( values.notna( ).sum( ) ) ) if 'oxygen' in daily.columns else ( 'date', 'size' ),
    )
)

print( 'daily rows:', len( daily ) )
print( 'station-year rows:', len( station_year_coverage ) )
display( daily.head( 3 ) )
display( station_year_coverage.head( 10 ) )

In [ ]:
# Keep the diagnostics light.
# One chart shows overall daily volume by year.
# One chart shows how complete station-years are.
year_counts = daily.groupby( 'year', as_index = False ).size( ).rename( columns = { 'size': 'n_rows' } )

fig, axes = plt.subplots( 1, 2, figsize = ( 16, 5 ) )

sns.lineplot( data = year_counts, x = 'year', y = 'n_rows', marker = 'o', ax = axes[ 0 ] )
axes[ 0 ].set_title( 'Phase 1 Daily Rows by Year' )
axes[ 0 ].set_xlabel( 'Year' )
axes[ 0 ].set_ylabel( 'Station-day rows' )

sns.histplot( data = station_year_coverage, x = 'n_temp_days', bins = 30, ax = axes[ 1 ] )
axes[ 1 ].set_title( 'Phase 1 Station-Year Water Temp Coverage' )
axes[ 1 ].set_xlabel( 'Observed water-temp days in year' )
axes[ 1 ].set_ylabel( 'Count of station-years' )

plt.tight_layout( )
plt.show( )

In [ ]:
# Save clean daily material for the next notebook.
# Keep the outputs plain CSV so each notebook can start fresh.
daily.to_csv( daily_out_path, index = False )
station_meta.to_csv( station_meta_out_path, index = False )
station_year_coverage.to_csv( station_year_out_path, index = False )

print( f'saved: {daily_out_path}' )
print( f'saved: {station_meta_out_path}' )
print( f'saved: {station_year_out_path}' )